In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from IPython.utils import io
with io.capture_output():
    !pip install unsloth -q
    !pip install datasets transformers accelerate peft trl -q
print("Установка завершена")

Установка завершена


In [2]:
# Название адаптера
LORA_OUTPUT = "govai_qwen35_4b_lora_v11"
os.environ["LORA_OUTPUT"] = LORA_OUTPUT

In [3]:
%%writefile train.py
import os
import gc
import json
import numpy as np
from unsloth import FastLanguageModel
import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

os.environ["TOKENIZERS_PARALLELISM"] = "false"
LORA_OUTPUT = os.environ["LORA_OUTPUT"]

def main():
    DATASET_PATH = '/kaggle/input/datasets/startexe/gov-analytics-dataset-v11/gov_analytics_dataset_v11.jsonl'
    MODEL_NAME = "unsloth/Qwen3.5-4B"
    MAX_SEQ_LENGTH = 2560
    LORA_R = 32
    LORA_ALPHA = LORA_R * 2
    LORA_DROPOUT = 0.05
    
    print(f"\nКонфигурация:")
    print(f"  LoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
    print(f"  Max seq length: {MAX_SEQ_LENGTH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        max_seq_length=MAX_SEQ_LENGTH,
    )
    
    if torch.cuda.current_device() == 0:
        model.print_trainable_parameters()

    SYSTEM = (
        "You are a specialized analyst for Russian government organizations. "
        "Analyze budgets by KOSGU codes, procurement according to 44-FZ and 223-FZ, "
        "develop strategies including PPP (115-FZ), energy efficiency (261-FZ), "
        "and personal data protection (152-FZ). "
        "Provide structured analysis with specific percentages, KPIs, and law citations. "
        "Respond in Russian with professional terminology."
    )

    def format_prompt(example):
        user = example['instruction']
        if example.get('input', '').strip():
            user += f"\n\nContext:\n{example['input']}"
        if tokenizer.chat_template:
            messages = [
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": user},
                {"role": "assistant", "content": example['output']}
            ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        else:
            text = (f"### System:\n{SYSTEM}\n\n### User:\n{user}\n\n"
                    f"### Assistant:\n{example['output']}{tokenizer.eos_token}")
        return {"text": text}

    data = []
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                if torch.cuda.current_device() == 0:
                    print(f"Ошибка в строке {line_num}: {e}")
    
    if torch.cuda.current_device() == 0:
        print(f"\nЗагружено примеров: {len(data)}")

    formatted = [format_prompt(d) for d in data]
    dataset = Dataset.from_list(formatted)
    
    split = dataset.train_test_split(test_size=0.1, seed=3407)
    train_dataset = split['train']
    eval_dataset = split['test']
    
    if torch.cuda.current_device() == 0:
        print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

    early_stopping = EarlyStoppingCallback(
        early_stopping_patience=3,
        early_stopping_threshold=0.001
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        args=SFTConfig(
            max_length=MAX_SEQ_LENGTH,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=16,
            warmup_steps=100,
            num_train_epochs=10,
            learning_rate=2e-5,
            logging_steps=10,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            output_dir="outputs",
            optim="adamw_8bit",
            seed=3407,
            dataset_text_field="text",
            report_to="none",
            lr_scheduler_type="cosine",
            weight_decay=0.01,
            max_grad_norm=0.3,
            ddp_find_unused_parameters=False,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        ),
        callbacks=[early_stopping],
    )

    trainer.train()

    if trainer.accelerator.is_main_process:
        print("\n" + "="*50)
        print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
        print("="*50)
        print(f"Лучшая модель выбрана по eval_loss")
        print(f"Финальный train loss: {next((x.get('loss') for x in reversed(trainer.state.log_history) if 'loss' in x), 'N/A')}")
        
        print("\nСохраняем модель...")
        model.save_pretrained(LORA_OUTPUT)
        tokenizer.save_pretrained(LORA_OUTPUT)
        print("Сохранено:", LORA_OUTPUT)
        print("Обучение и сохранение завершено!")

if __name__ == "__main__":
    main()

Writing train.py


In [4]:
!torchrun --nproc_per_node=2 train.py

W0614 15:17:21.397000 129 torch/distributed/run.py:852] 
W0614 15:17:21.397000 129 torch/distributed/run.py:852] *****************************************
W0614 15:17:21.397000 129 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0614 15:17:21.397000 129 torch/distributed/run.py:852] *****************************************
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!

Конфигурация:
  LoRA: r=32, alpha=64, dropout=0.05
  Max seq length: 2560
Конфигурация:

  LoRA: r=32, alpha=64, dropout=0.05
  Max seq length: 2560
==((====))==  Unsloth 2026.6.7: Fast Qwen3_5 patching. 

In [5]:
import shutil
import os
from IPython.display import FileLink

if os.path.exists(LORA_OUTPUT):
    shutil.make_archive(LORA_OUTPUT, 'zip', LORA_OUTPUT)
    print(f"Архив создан: {LORA_OUTPUT}.zip ({os.path.getsize(LORA_OUTPUT + '.zip')/1024/1024:.1f} MB)")
else:
    print("Ошибка: папка с моделью не найдена!")

Архив создан: govai_qwen35_4b_lora_v11.zip (153.7 MB)


In [6]:
FileLink(LORA_OUTPUT + ".zip")

/kaggle/working/govai_qwen35_4b_lora_v11.zip

In [7]:
from unsloth import FastLanguageModel
import torch
import re

# 1. Загружаем модель и адаптер
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = LORA_OUTPUT,
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Тестовый запрос
prompt = "Проанализируй бюджет организации на закупку оборудования (КОСГУ 310) в размере 10 млн руб. Какие риски по 44-ФЗ при сокращении лимитов на 15%?"

# System prompt — тот же, что при обучении
SYSTEM = ("You are a specialized analyst for government organizations. "
          "Analyze budgets by KOSGU, procurement according to 44-FZ, "
          "develop development strategies for budget directions. "
          "Use pandas and seaborn for visualization. "
          "Respond in Russian with structured analysis.")

messages = [
    {
        "role": "system", 
        "content": [{"type": "text", "text": SYSTEM}]
    },
    {
        "role": "user", 
        "content": [{"type": "text", "text": prompt}]
    }
]

# 3. Подготовка входа
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 4. Генерация ответа
outputs = model.generate(
    input_ids = inputs, 
    max_new_tokens = 1024, 
    use_cache = True,
    temperature = 1.0,
    top_p = 0.95,
    min_p=0.0,
    repetition_penalty=1.0,
    do_sample=True
)

raw_response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)

print("="*60)
print("СЫРОЙ ОТВЕТ МОДЕЛИ (с thinking mode):")
print("="*60)
print(raw_response)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


СЫРОЙ ОТВЕТ МОДЕЛИ (с thinking mode):
The user wants me to analyze a budget for equipment procurement (COSG 310) in the amount of 10 million rubles. They want to know about risks according to 44-FZ if the limit is reduced by 15%.

I should:
1. Analyze the budget allocation according to COSG (Class of Subjects/Government Organization)
2. Apply procurement rules from 44-FZ
3. Use pandas and seaborn for visualization
4. Respond in Russian with structured analysis

Let me analyze the provided budget data and create relevant visualizations.

Key points for 44-FZ procurement:
- Threshold for public procurement: 1 million rubles
- Competition requirements for certain values
- Payment terms: typically 30-90 days
- Budget reduction may require new tender requirements or may limit options

COSG 310 is for equipment and machinery.

I'll analyze the budget and show the risks with visualizations.
</think>

## Анализ Бюджета: Закупка оборудования (КОСГУ 310)

**Исходные данные:**
- Бюджет: 10 000 00